In [1]:
import torch
import librosa
import numpy as np
import parselmouth  # Para extração de pitch (Praat)
import pandas as pd
from spafe.features.mfcc import mfcc
from spafe.features.lfcc import lfcc
from spafe.features.cqcc import cqcc
from spafe.features.rplp import rplp

def extract_features(audio_path, sample_rate=16000):
    """
    Extrai características de um arquivo de áudio.

    Parâmetros:
        audio_path (str): Caminho do arquivo .wav
        sample_rate (int): Taxa de amostragem padrão 16kHz.

    Retorno:
        features (dict): Dicionário contendo os recursos extraídos.
    """
    # Carregar áudio
    y, sr = librosa.load(audio_path, sr=sample_rate, mono=True)

    ### 1️⃣ MFCC (Mel-Frequency Cepstral Coefficients)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13).mean(axis=1)

    ### 2️⃣ i-MFCC (Instantaneous MFCC) - derivadas
    delta_mfcc = librosa.feature.delta(mfccs)
    delta2_mfcc = librosa.feature.delta(mfccs, order=2)

    ### 3️⃣ CQCC (Constant Q Cepstral Coefficients)
    cqcc_features = cqcc(sig=y, fs=sr, num_ceps=13).mean(axis=1)

    ### 4️⃣ LFO (Low-Frequency Oscillations)
    lfo = librosa.feature.rms(y=y).mean()  # Root Mean Square Energy

    ### 5️⃣ Pitch (F0 - Fundamental Frequency) usando Praat (Parselmouth)
    snd = parselmouth.Sound(audio_path)
    pitch_values = snd.to_pitch().selected_array['frequency']
    pitch_values[pitch_values == 0] = np.nan  # Remover valores zero
    pitch_mean = np.nanmean(pitch_values)  # Média ignorando NaNs

    ### 6️⃣ LFCC (Linear Frequency Cepstral Coefficients)
    lfcc_features = lfcc(sig=y, fs=sr, num_ceps=13).mean(axis=1)

    ### 7️⃣ RPLP (Perceptual Linear Prediction)
    rplp_features = rplp(sig=y, fs=sr).mean(axis=1)

    ### 8️⃣ Energy-based Features (Zero-Crossing Rate)
    zcr = librosa.feature.zero_crossing_rate(y).mean()

    # Criar um dicionário com os recursos extraídos, garantindo que TODOS sejam arrays
    features = {
        "MFCC": mfccs,
        "i-MFCC (Δ)": delta_mfcc,
        "i-MFCC (Δ²)": delta2_mfcc,
        "CQCC": cqcc_features,
        "LFO (RMS Energy)": np.array([lfo]),  # Converte escalar para array
        "Pitch (F0)": np.array([pitch_mean]),  # Converte escalar para array
        "LFCC": lfcc_features,
        "RPLP": rplp_features,
        "Zero-Crossing Rate": np.array([zcr]),  # Converte escalar para array
    }

    return features

# Caminho do arquivo de áudio
audio_path = "/DATA2/MyWorkspaces/PyWorkspaces/AudioTransformers/notebooks/teste2.wav"  # Substitua pelo caminho do seu arquivo

# Extração das características
features = extract_features(audio_path)

# Criar um DataFrame para visualizar melhor
df_features = pd.DataFrame.from_dict(features, orient='index').T
print(df_features)


/tmp/ipykernel_29731/238931068.py:23: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_path, sr=sample_rate, mono=True)
/home/alex/miniconda3/envs/AudioTransformers/lib/python3.9/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


FileNotFoundError: [Errno 2] No such file or directory: '/DATA2/MyWorkspaces/PyWorkspaces/AudioTransformers/notebooks/teste2.wav'

In [26]:
features['MFCC'].shape

(13,)

In [3]:
import torch
import librosa
import numpy as np
import parselmouth  # Para extração de pitch (Praat)
import pandas as pd
from spafe.features.mfcc import mfcc
from spafe.features.lfcc import lfcc
from spafe.features.cqcc import cqcc
from spafe.features.rplp import rplp

# --------------------------------------------
# 1️⃣ Função para carregar o áudio
# --------------------------------------------
def load_audio(audio_path, sample_rate=16000):
    y, sr = librosa.load(audio_path, sr=sample_rate, mono=True)
    return y, sr

# --------------------------------------------
# 2️⃣ Extração de MFCC
# --------------------------------------------
def extract_mfcc(y, sr, num_ceps=13):
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=num_ceps)
    delta_mfcc = librosa.feature.delta(mfccs)
    delta2_mfcc = librosa.feature.delta(mfccs, order=2)
    return mfccs.mean(axis=1), delta_mfcc.mean(axis=1), delta2_mfcc.mean(axis=1)

# --------------------------------------------
# 3️⃣ Extração de CQCC
# --------------------------------------------
def extract_cqcc(y, sr, num_ceps=13):
    cqcc_features = cqcc(sig=y, fs=sr, num_ceps=num_ceps)
    return cqcc_features.mean(axis=1)

# --------------------------------------------
# 4️⃣ Extração de LFO (RMS Energy)
# --------------------------------------------
def extract_lfo(y):
    return librosa.feature.rms(y=y).mean()

# --------------------------------------------
# 5️⃣ Extração de Pitch (F0) com Praat (Parselmouth)
# --------------------------------------------
def extract_pitch(audio_path):
    snd = parselmouth.Sound(audio_path)
    pitch_values = snd.to_pitch().selected_array['frequency']
    pitch_values[pitch_values == 0] = np.nan  # Remover valores zero
    return np.nanmean(pitch_values)  # Média ignorando NaNs

# --------------------------------------------
# 6️⃣ Extração de LFCC (Linear Frequency Cepstral Coefficients)
# --------------------------------------------
def extract_lfcc(y, sr, num_ceps=13):
    lfcc_features = lfcc(sig=y, fs=sr, num_ceps=num_ceps)
    return lfcc_features.mean(axis=1)

# --------------------------------------------
# 7️⃣ Extração de RPLP (Perceptual Linear Prediction)
# --------------------------------------------
def extract_rplp(y, sr):
    rplp_features = rplp(sig=y, fs=sr)
    return rplp_features.mean(axis=1)

# --------------------------------------------
# 8️⃣ Extração de Zero-Crossing Rate (ZCR)
# --------------------------------------------
def extract_zcr(y):
    return librosa.feature.zero_crossing_rate(y).mean()

# --------------------------------------------
# 9️⃣ Função principal para extrair todas as características
# --------------------------------------------
def extract_all_features(audio_path):
    """
    Extrai todas as características do áudio.

    Parâmetros:
        audio_path (str): Caminho do arquivo .wav

    Retorno:
        features (dict): Dicionário contendo os recursos extraídos.
    """
    # Carregar o áudio
    y, sr = load_audio(audio_path)

    # Extrair todas as características
    mfccs, delta_mfcc, delta2_mfcc = extract_mfcc(y, sr)
    cqcc_features = extract_cqcc(y, sr)
    lfo = extract_lfo(y)
    pitch_mean = extract_pitch(audio_path)
    lfcc_features = extract_lfcc(y, sr)
    rplp_features = extract_rplp(y, sr)
    zcr = extract_zcr(y)

    # Criar dicionário com os recursos extraídos
    features = {
        "MFCC": mfccs,
        "i-MFCC (Δ)": delta_mfcc,
        "i-MFCC (Δ²)": delta2_mfcc,
        "CQCC": cqcc_features,
        "LFO (RMS Energy)": np.array([lfo]),  # Converte escalar para array
        "Pitch (F0)": np.array([pitch_mean]),  # Converte escalar para array
        "LFCC": lfcc_features,
        "RPLP": rplp_features,
        "Zero-Crossing Rate": np.array([zcr]),  # Converte escalar para array
    }

    return features

# --------------------------------------------
# 🔹 Teste com um arquivo de áudio
# --------------------------------------------
audio_path = "teste2.wav"  # Substitua pelo caminho do seu arquivo

# Extração das características
features = extract_all_features(audio_path)

# Criar um DataFrame para visualizar melhor
df_features = pd.DataFrame.from_dict(features, orient='index').T
print(df_features)


           MFCC  i-MFCC (Δ)  i-MFCC (Δ²)       CQCC  LFO (RMS Energy)  \
0   -323.051239   -0.040741    -0.091294 -12.332305          0.009571   
1     92.348099    0.072892    -0.001003 -12.513104               NaN   
2     10.851058    0.007489     0.047305 -11.858686               NaN   
3      6.653301   -0.012731    -0.013605 -12.926205               NaN   
4      3.240074    0.059239    -0.014158 -12.253154               NaN   
..          ...         ...          ...        ...               ...   
437         NaN         NaN          NaN -12.469459               NaN   
438         NaN         NaN          NaN -12.779902               NaN   
439         NaN         NaN          NaN -12.371883               NaN   
440         NaN         NaN          NaN -12.435880               NaN   
441         NaN         NaN          NaN -13.322413               NaN   

     Pitch (F0)      LFCC      RPLP  Zero-Crossing Rate  
0    116.443334 -4.269862 -5.552734            0.129398  
1      

In [29]:
y, sr = load_audio(audio_path)
a=extract_mfcc(y, sr, num_ceps=13)

In [30]:
len(y)/16000

4.443875